# PV100F + PV4026 50张分组蒸馏验证

本 notebook 只运行 GitHub 已提交代码。测试集包含 20 张常规正样本、15 张小掩膜困难正样本和 15 张 PV100F 负样本，用于验证 InternVL Teacher 特征缓存与图像级语义判断，不训练 Student。

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('当前 Kaggle notebook 未分配 GPU，请确认 kernel-metadata.json 中 enable_gpu=true')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
repo_dir = Path('/kaggle/working/internvl-pv-distillation')
repo_url = 'https://github.com/Zh0416/internvl-pv-distillation.git'
fixed_commit = 'fe2476b7ad337bcd52e0f3aaf8a5188f04295dab'

if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', fixed_commit], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != fixed_commit:
    raise RuntimeError(f'Git commit不一致: expected={fixed_commit}, actual={actual_commit}')
print('Repository commit:', actual_commit)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(repo_dir / 'requirements.txt')],
    check=True,
)
os.chdir(repo_dir)

In [ ]:
config_path = repo_dir / 'configs/kaggle_pv_50_distill.yaml'
mounted_dirs = sorted(
    str(path) for path in Path('/kaggle/input').glob('**/*')
    if path.is_dir() and len(path.relative_to('/kaggle/input').parts) <= 4
)
print('Mounted input directories:', json.dumps(mounted_dirs, ensure_ascii=False, indent=2))

subprocess.run(
    [sys.executable, 'scripts/05_small_distillation_test.py', '--config', str(config_path), '--selection-only'],
    check=True,
)
dataset_report_path = Path('/kaggle/working/outputs/reports/dataset_report.json')
selection_path = Path('/kaggle/working/outputs/reports/selection_manifest.json')
dataset_report = json.loads(dataset_report_path.read_text(encoding='utf-8'))
selection = json.loads(selection_path.read_text(encoding='utf-8'))
print(json.dumps({
    'total_images': dataset_report['total_images'],
    'positive_count': dataset_report['positive_count'],
    'negative_count': dataset_report['negative_count'],
    'missing_mask_count': dataset_report['missing_mask_count'],
    'image_mask_one_to_one': dataset_report['image_mask_one_to_one'],
    'selected_count': selection['selected_count'],
}, ensure_ascii=False, indent=2))

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/05_small_distillation_test.py', '--config', str(config_path)],
    check=True,
)

In [ ]:
report_path = Path('/kaggle/working/outputs/reports/small_distillation_report.json')
report = json.loads(report_path.read_text(encoding='utf-8'))
summary = {
    'repository_commit': fixed_commit,
    'training_performed': report['training_performed'],
    'parameters': report['parameters'],
    'metrics': report['metrics'],
    'metrics_by_group': report['metrics_by_group'],
    'single_test': {
        'cache_key': report['single_test']['sample']['cache_key'],
        'ground_truth': report['single_test']['ground_truth'],
        'prediction': report['single_test']['prediction'],
        'confidence': report['single_test']['semantic']['confidence'],
        'correct': report['single_test']['correct'],
    },
    'runtime': report['runtime'],
    'remaining_issues': [
        '本次指标是图像级光伏存在性指标，不是像素级分割指标。',
        '本次只验证Teacher缓存与语义标签，尚未训练Student。',
        '50张分组测试仍不是完整独立测试集，指标用于Teacher方案筛选。',
    ],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Full report:', report_path)